## **Creacion de Esquemas**

In [0]:
ESQUEMAS = {
    "cine_bronze": "Capa Bronze: datos tal como llegaron del origen, sin transformar",
    "cine_silver": "Capa Silver: datos limpios, tipados y con reglas de calidad",
    "cine_gold":   "Capa Gold: tablas agregadas listas para consumo de negocio",
}

for nombre, descripcion in ESQUEMAS.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS workspace.{nombre}
        COMMENT '{descripcion}'
    """)
    print("Esquema listo: workspace." + nombre)

## Creacion del Volumen

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.cine_bronze.aterrizaje
COMMENT 'Zona de aterrizaje: archivos originales descargados del origen'

## Lectura de Datos

In [0]:
import urllib.request
URL = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/movies.json"
DESTINO = "/Volumes/workspace/cine_bronze/aterrizaje/movies.json"

with urllib.request.urlopen(URL, timeout=60) as respuesta:
    contenido = respuesta.read()

with open(DESTINO, "wb") as f:
    f.write(contenido)

print(f"Descargados {len(contenido):,} bytes")
print("Guardado en:", DESTINO)

## Insertar datos en Bronce

In [0]:
df_crudo = (spark.read
    .option("multiLine", True)
    .json(DESTINO))
df_crudo.printSchema()
print('Filas:',df_crudo.count())
df_crudo.show(10)

## Nombres de Columna

Se ajustan los nombres de columnas para que no tengan espacios

In [0]:
import re
from pyspark.sql import functions as F

def sanear(nombre):
    """US Gross -> us_gross   |   Running Time min -> running_time_min"""
    return re.sub(r"[^0-9a-zA-Z]+", "_", nombre).strip("_").lower()

nombres = [sanear(c) for c in df_crudo.columns] + ["_archivo_origen", "_ingestado_en"]

df_bronze = (df_crudo
    .withColumn("_archivo_origen", F.col("_metadata.file_path"))
    .withColumn("_ingestado_en",   F.current_timestamp())
    .toDF(*nombres))

print(df_bronze.columns)

## Crear tabla en bronce

In [0]:
(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("workspace.cine_bronze.peliculas_crudo"))

print("Tabla creada: workspace.cine_bronze.peliculas_crudo")

## Capa Silver: tipar, nombrar y decidir
Silver hace tres cosas:

Tipos de verdad — las fechas dejan de ser texto, los importes dejan de ser genéricos.
Nombres de negocio — quien consulte esta tabla habla español y habla de negocio, no de columnas de un JSON ajeno.
Reglas de calidad — se decide qué fila es válida y qué fila no

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_tipadas AS
SELECT
  title                                       AS titulo,
  try_to_date(release_date, 'MMM dd yyyy')    AS fecha_estreno,
  major_genre                                 AS genero,
  mpaa_rating                                 AS clasificacion,
  distributor                                 AS estudio,
  director                                    AS director,
  creative_type                               AS tipo_creativo,
  source                                      AS origen_historia,
  CAST(running_time_min AS INT)               AS duracion_min,
  CAST(production_budget AS BIGINT)           AS presupuesto,
  nullif(CAST(us_gross AS BIGINT), 0)         AS recaudacion_eeuu,
  nullif(CAST(worldwide_gross AS BIGINT), 0)  AS recaudacion_mundial,
  CAST(imdb_rating AS DOUBLE)                 AS calificacion_imdb,
  CAST(imdb_votes AS BIGINT)                  AS votos_imdb,
  CAST(rotten_tomatoes_rating AS INT)         AS calificacion_rt,
  _ingestado_en
FROM workspace.cine_bronze.peliculas_crudo

## Segundo punto de la tarea
Se adiciona como validacion si el genero o el director es nulo

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW peliculas_evaluadas AS
SELECT *,
  array_compact(array(
    CASE WHEN titulo IS NULL                  THEN 'titulo nulo' END,
    CASE WHEN fecha_estreno IS NULL           THEN 'fecha ilegible' END,
    CASE WHEN fecha_estreno > current_date()  THEN 'fecha futura' END,
    CASE WHEN genero IS NULL                  THEN 'genero nulo' END,
    CASE WHEN director IS NULL                  THEN 'director nulo' END,
    CASE WHEN presupuesto IS NULL
           OR presupuesto <= 0                THEN 'presupuesto no positivo' END
  )) AS motivos_rechazo
FROM peliculas_tipadas

## Visualizar filas erradas

In [0]:
%sql
SELECT motivo, count(*) AS filas
FROM (
  SELECT explode(motivos_rechazo) AS motivo
  FROM peliculas_evaluadas
  WHERE size(motivos_rechazo) > 0
)
GROUP BY motivo
ORDER BY filas DESC

## Crear tabla de cuarentena

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas_cuarentena
COMMENT 'Filas que no superaron las reglas de calidad de la capa Silver'
AS
SELECT *, current_timestamp() AS _cuarentena_en
FROM peliculas_evaluadas
WHERE size(motivos_rechazo) > 0

## Crear la tabla en Silver

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_silver.peliculas
COMMENT 'Películas limpias y tipadas. Una fila por título y fecha de estreno'
AS
SELECT * EXCEPT (motivos_rechazo, _fila)
FROM (
  SELECT *,
    row_number() OVER (
      PARTITION BY titulo, fecha_estreno
      ORDER BY recaudacion_mundial DESC NULLS LAST) AS _fila
  FROM peliculas_evaluadas
  WHERE size(motivos_rechazo) = 0
)
WHERE _fila = 1

## Verificar que las cuentas cuadren

In [0]:
%sql
SELECT
  (SELECT count(*) FROM workspace.cine_bronze.peliculas_crudo)      AS bronze,
  (SELECT count(*) FROM workspace.cine_silver.peliculas)            AS silver,
  (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena) AS cuarentena,
  (SELECT count(*) FROM workspace.cine_silver.peliculas)
    + (SELECT count(*) FROM workspace.cine_silver.peliculas_cuarentena) AS suma_comprobacion

## Contrato tabla Silver

In [0]:
CONTRATO = [
    "ALTER COLUMN titulo SET NOT NULL",
    "ALTER COLUMN fecha_estreno SET NOT NULL",
    "ADD CONSTRAINT presupuesto_positivo CHECK (presupuesto > 0)",
    "ADD CONSTRAINT duracion_razonable "
    "  CHECK (duracion_min IS NULL OR duracion_min BETWEEN 1 AND 500)",
    "ADD CONSTRAINT imdb_en_rango "
    "  CHECK (calificacion_imdb IS NULL OR calificacion_imdb BETWEEN 0 AND 10)",
]

for clausula in CONTRATO:
    spark.sql(f"ALTER TABLE workspace.cine_silver.peliculas {clausula}")
    print("Aplicado:", clausula.split("CHECK")[0].strip())

In [0]:
%sql
ALTER TABLE workspace.cine_silver.peliculas
ADD CONSTRAINT pk_peliculas PRIMARY KEY (titulo, fecha_estreno)

In [0]:
%sql
Select * from workspace.cine_silver.peliculas

## Crear tabla en GOLD

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.cine_gold.peliculas_rentables
COMMENT 'Peliculas rentables con mejor retorno'
AS
SELECT
  titulo,
  genero,
  votos_imdb,
  recaudacion_mundial,
  presupuesto,
  CAST(try_divide(recaudacion_mundial,presupuesto) AS int )  AS retorno
FROM workspace.cine_silver.peliculas
WHERE votos_imdb > 10000
ORDER BY retorno DESC
LIMIT 50


In [0]:
%sql
Select * from workspace.cine_gold.peliculas_rentables

## 3-Documentar la tabla
Punto 3 de la tarea


In [0]:
COMENTARIOS = {
    "titulo":              "Título original de la película. Parte de la clave de negocio",
    "genero":              "Fecha de estreno en salas. Parte de la clave de negocio",
    "votos_imdb":          "Votos obtenidos por la pelicula",
    "recaudacion_mundial": "Taquilla mundial en dólares. NULL si el origen no la reportó",
    "presupuesto":         "Presupuesto de producción en dólares. Dato financiero confidencial",
    "retorno":             "Retorno en utilidad de la pelicula",
}
for columna, texto in COMENTARIOS.items():
    spark.sql(f"ALTER TABLE workspace.cine_gold.peliculas_rentables "
              f"ALTER COLUMN {columna} COMMENT '{texto}'")
    print("Documentada:", columna)

In [0]:
%sql
DESCRIBE TABLE workspace.cine_gold.peliculas_rentables

In [0]:
ETIQUETAS = [
    "ALTER TABLE workspace.cine_silver.peliculas "
    "SET TAGS ('capa' = 'silver', 'dominio' = 'cine', 'certificada' = 'si')",
    "ALTER TABLE workspace.cine_gold.peliculas_rentables ALTER COLUMN retorno "
    "SET TAGS ('sensibilidad' = 'confidencial')",    
    "ALTER TABLE workspace.cine_gold.peliculas_rentables "
    "SET TAGS ('capa' = 'gold', 'dominio' = 'cine')",
]

for sentencia in ETIQUETAS:
    spark.sql(sentencia)

print("Etiquetas aplicadas")

In [0]:
%sql
SELECT table_name, column_name, tag_name, tag_value
FROM workspace.information_schema.column_tags
WHERE schema_name = 'cine_gold'

In [0]:
PERMISOS = [
    "GRANT USE SCHEMA ON SCHEMA workspace.cine_gold TO `account users`",
    "GRANT SELECT     ON SCHEMA workspace.cine_gold TO `account users`",
]

for sentencia in PERMISOS:
    spark.sql(sentencia)
    print("Concedido:", sentencia.split(" ON ")[0])